In [10]:
library(tidyr)
library(dplyr)
library(ggplot2)
library(patchwork)
library(data.table)

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/module_projection_fxns.R")

Here I visualize module genes to try to decide on how to clean up the bulk data prior to running FM again (to hopefully boost signal of certain cell types)

In [11]:
options(repr.plot.width=20, repr.plot.height=8, repr.plot.res=150)

In [12]:
mod_def <- "TopModPosFDR"

### Bulk data and enrichments

In [ ]:
# *** Bulk gene expression should be the same dataset used in to find modules and enrichments ***

bulk_data_source <- "hahn_2023_cortex_STAR_TPM_All_94_outliers_removed"
bulk_expr <- fread("hahn_2023_cortex_STAR_TPM_SampleNetworks/All_09-53-50/hahn_2023_cortex_STAR_TPM_All_94_outliers_removed.csv", data.table=FALSE)
colnames(bulk_expr)[1] <- "Gene"

enrichment_source <- ""

top_mods_df <- fread(paste0("data/enrichments/", enrichment_source, ".csv"), data.table=FALSE)

max_qval <- 1

top_mods_df <- top_mods_df[!is.na(top_mods_df$Qval),]
top_mods_df_subset <- top_mods_df[top_mods_df$Qval < max_qval,]

dim(top_mods_df_subset)

[1] 21  7

### Prep single cell data

In [ ]:
# sc_data_source <- "yao_2021_MOp_STAR_gene_counts_normalized"

# sc_expr <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/yao_2021/MOp/yao_2021_MOp_STAR_gene_counts.csv", data.table=FALSE)
# sampleinfo <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/yao_2021/MOp/yao_2021_MOp_STAR_sampleinfo.csv", data.table=FALSE)
# colnames(sc_expr)[1] <- "Gene"

# # Remove cell types with only a few cells

# min_cells <- 5

# ctype_tally <- table(sampleinfo$subclass_label)
# ctypes_to_keep <- names(ctype_tally)[ctype_tally >= min_cells]
# cells_to_keep <- which(sampleinfo$subclass_label %in% ctypes_to_keep)

# sc_expr <- sc_expr[, c(1, cells_to_keep + 1)]
# sampleinfo <- sampleinfo[cells_to_keep,]

# all.equal(colnames(sc_expr)[-1], sampleinfo$Cell_ID)

# total_expr <- colSums(sc_expr[,-1])
# sc_expr_norm <- sweep(sc_expr[,-1], MARGIN=2, FUN="/", STATS=total_expr) * 1e4
# sc_expr_norm <- data.frame(Gene=sc_expr[,1], sc_expr_norm) 

# ctype_assignment_vec <- sampleinfo$subclass_label

[1] TRUE

In [6]:
sc_data_source <- "yao_2021_ACA_STAR_gene_counts_normalized"

sc_expr <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/yao_2021/ACA/yao_2021_ACA_STAR_gene_counts.csv", data.table=FALSE)
sampleinfo <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/yao_2021/ACA/yao_2021_ACA_STAR_sampleinfo.csv", data.table=FALSE)
colnames(sc_expr)[1] <- "Gene"

# Remove cell types with only a few cells

min_cells <- 5

ctype_tally <- table(sampleinfo$subclass_label)
ctypes_to_keep <- names(ctype_tally)[ctype_tally >= min_cells]
cells_to_keep <- which(sampleinfo$subclass_label %in% ctypes_to_keep)

sc_expr <- sc_expr[, c(1, cells_to_keep + 1)]
sampleinfo <- sampleinfo[cells_to_keep,]

all.equal(colnames(sc_expr)[-1], sampleinfo$Cell_ID)

total_expr <- colSums(sc_expr[,-1])
sc_expr_norm <- sweep(sc_expr[,-1], MARGIN=2, FUN="/", STATS=total_expr) * 1e4
sc_expr_norm <- data.frame(Gene=sc_expr[,1], sc_expr_norm) 

ctype_assignment_vec <- sampleinfo$subclass_label

[1] TRUE

## Visualize module genes

In [32]:
outdir <- paste0("figures/module_genes/", sc_data_source, "/", mod_def)

if (!dir.exists(outdir)) {
    dir.create(outdir, recursive=TRUE)
}

filename <- paste0(outdir, "/", enrichment_source, ".pdf")

max_genes <- 15

pdf(file=filename)

for (i in 1:nrow(top_mods_df_subset)) {
    print(paste("Module", i))

    mod <- top_mods_df_subset$Module[i]
    kME_path <- top_mods_df_subset$kME_path[i]
    mod_genes <- get_mod_genes(kME_path, mod, mod_def)

    plot_title <- paste(
        top_mods_df_subset$Cell_type[i], mod_def, 
        "\n", mod, top_mods_df_subset$Network[i]
    ) 
    plot_sub <- paste("Qval:", round(top_mods_df_subset$Qval[i], 4))

    # Plot gene expression over bulk samples
    
    mod_genes_subset <- na.omit(mod_genes[1:max_genes])
    plot_gene_expr_over_samples(bulk_expr, mod_genes_subset, plot_title, plot_sub, target_species=NULL)

    # Plot module genes in single cell data

    plot_gene_projections(sc_expr_norm, mod_genes, ctype_assignment_vec, plot_title, plot_sub, target_species="mouse")
}

dev.off()

[1] "Module 1"


Warning message:
“There was 1 warning in `filter()`.
ℹ In argument: `DB.Class.Key == class_key & Common.Organism.Name == "mouse,
  laboratory"`.
Caused by warning in `DB.Class.Key == class_key`:
! longer object length is not a multiple of shorter object length”
Warning message:
“There was 1 warning in `filter()`.
ℹ In argument: `DB.Class.Key == class_key & Common.Organism.Name == "mouse,
  laboratory"`.
Caused by warning in `DB.Class.Key == class_key`:
! longer object length is not a multiple of shorter object length”
Warning message:
“There was 1 warning in `filter()`.
ℹ In argument: `DB.Class.Key == class_key & Common.Organism.Name == "mouse,
  laboratory"`.
Caused by warning in `DB.Class.Key == class_key`:
! longer object length is not a multiple of shorter object length”
Warning message:
“There was 1 warning in `filter()`.
ℹ In argument: `DB.Class.Key == class_key & Common.Organism.Name == "mouse,
  laboratory"`.
Caused by warning in `DB.Class.Key == class_key`:
! longer object len

[1] "Module 2"


Warning message:
“There was 1 warning in `filter()`.
ℹ In argument: `DB.Class.Key == class_key & Common.Organism.Name == "mouse,
  laboratory"`.
Caused by warning in `DB.Class.Key == class_key`:
! longer object length is not a multiple of shorter object length”
Warning message:
“There was 1 warning in `filter()`.
ℹ In argument: `DB.Class.Key == class_key & Common.Organism.Name == "mouse,
  laboratory"`.
Caused by warning in `DB.Class.Key == class_key`:
! longer object length is not a multiple of shorter object length”
Warning message:
“There was 1 warning in `filter()`.
ℹ In argument: `DB.Class.Key == class_key & Common.Organism.Name == "mouse,
  laboratory"`.
Caused by warning in `DB.Class.Key == class_key`:
! longer object length is not a multiple of shorter object length”
Warning message:
“There was 1 warning in `filter()`.
ℹ In argument: `DB.Class.Key == class_key & Common.Organism.Name == "mouse,
  laboratory"`.
Caused by warning in `DB.Class.Key == class_key`:
! longer object len

[1] "Module 3"


...

[1] "Module 37"
[1] "Module 38"


Warning message:
“There was 1 warning in `filter()`.
ℹ In argument: `DB.Class.Key == class_key & Common.Organism.Name == "mouse,
  laboratory"`.
Caused by warning in `DB.Class.Key == class_key`:
! longer object length is not a multiple of shorter object length”
Warning message:
“There was 1 warning in `filter()`.
ℹ In argument: `DB.Class.Key == class_key & Common.Organism.Name == "mouse,
  laboratory"`.
Caused by warning in `DB.Class.Key == class_key`:
! longer object length is not a multiple of shorter object length”
Warning message:
“There was 1 warning in `filter()`.
ℹ In argument: `DB.Class.Key == class_key & Common.Organism.Name == "mouse,
  laboratory"`.
Caused by warning in `DB.Class.Key == class_key`:
! longer object length is not a multiple of shorter object length”
Warning message:
“There was 1 warning in `filter()`.
ℹ In argument: `DB.Class.Key == class_key & Common.Organism.Name == "mouse,
  laboratory"`.
Caused by warning in `DB.Class.Key == class_key`:
! longer object len

[1] "Module 39"


Warning message:
“There was 1 warning in `filter()`.
ℹ In argument: `DB.Class.Key == class_key & Common.Organism.Name == "mouse,
  laboratory"`.
Caused by warning in `DB.Class.Key == class_key`:
! longer object length is not a multiple of shorter object length”


agg_record_790163007 
                   2

In [18]:
dev.off()

pdf 
  3

In [19]:
enrichment_source

[1] "GTEx_cortex_counts_TMMF_All_501_outliers_removed_mergeParam0.95_subsetCutoff10.171_Modules_MO_1271sets_enrichments_top_Qval_mods"

## Visualize marker genes

In [8]:
marker_source <- "Claude_marker_genes"

marker_genes_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/marker_genes/AI/Claude_cortical_markers_mouse.RDS")

In [16]:
outdir <- paste0("figures/marker_genes/", marker_source, "/", sc_data_source)

if (!dir.exists(outdir)) {
    dir.create(outdir, recursive=TRUE)
}

filename <- paste0(outdir, "/", bulk_data_source, ".pdf")

pdf(file=filename)

for (i in seq_along(marker_genes_list)) {
    gene_vec <- marker_genes_list[[i]]

    plot_title <- names(marker_genes_list)[i] 
    plot_sub <- NULL

    plot_gene_expr_over_samples(bulk_expr, gene_vec[1:15], plot_title, plot_sub)
    plot_gene_projections(sc_expr_norm, gene_vec, ctype_assignment_vec, plot_title, plot_sub, target_species="mouse")
}

dev.off()

agg_record_1238269973 
                    2

In [51]:
dev.off()

pdf 
  3

# Markers vs. modules

In [ ]:
# source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/top_corr_module_fxns.R")

# # Get genes for each top module, and intersect with marker genes; which modules are these genes appearing in?

# marker_genes <- unlist(human_marker_genes_list)

# # inds <- c()
# overlap_genes <- vector(mode="list", length=nrow(top_mods_df_subset))
# names(overlap_genes) <- paste(top_mods_df_subset$Cell_type, top_mods_df_subset$Qval)

# for (i in 1:nrow(top_mods_df_subset)) {
#     mod <- top_mods_df_subset$Module[i]
#     kME_path <- top_mods_df_subset$kME_path[i]
#     mod_genes <- get_mod_genes(kME_path, mod, mod_def)
#     set_overlap_genes <- intersect(mod_genes, marker_genes)
#     if (length(set_overlap_genes) > 0) {
#       overlap_genes[[i]] <- set_overlap_genes 
#     }
# }